In [ ]:
# 誤差逆伝播法

In [12]:
# 乗算レイヤ
class MulLayer:
    def __init__(self):
        self.x = None
        self.y = None

    def forward(self, x, y):
        self.x = x
        self.y = y
        out = x * y
        return out

    def backward(self, dout):
        dx = dout * self.y # 乗算の式をxで偏微分すると self.y が残るので、その値と dout を乗算
        dy = dout * self.x
        return dx, dy

In [19]:
apple = 100
apple_num = 2
tax = 1.1

mul_apple_layer = MulLayer()
mul_tax_layer = MulLayer()

# forward（順伝播）
apple_price = mul_apple_layer.forward(apple, apple_num)
price = mul_tax_layer.forward(apple_price, tax)
print(round(price))

# backward（逆伝播）
dprice = 1
dapple_price, dtax = mul_tax_layer.backward(dprice)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)
print(round(dapple), round(dapple_num), dtax)

220
2 110 200


In [14]:
# 加算レイヤ
class AddLayer:
    def __init__(self):
        pass

    def forward(self, x, y):
        out = x + y
        return out

    def backward(self, dout):
        dx = dout * 1
        dy = dout * 1
        return dx, dy

In [16]:
apple = 100
apple_num = 2
orange = 150
orange_num = 3
tax = 1.1

# layer
mul_apple_layer = MulLayer()
mul_orange_layer = MulLayer()
add_apple_orange_layer = AddLayer()
mul_tax_layer = MulLayer()

# forward
apple_price = mul_apple_layer.forward(apple, apple_num)  # (1)
orange_price = mul_orange_layer.forward(orange, orange_num)  # (2)
all_price = add_apple_orange_layer.forward(apple_price, orange_price)  # (3)
price = mul_tax_layer.forward(all_price, tax)  # (4)

# backward
dprice = 1
dall_price, dtax = mul_tax_layer.backward(dprice)  # (4)
dapple_price, dorange_price = add_apple_orange_layer.backward(dall_price)  # (3)
dorange, dorange_num = mul_orange_layer.backward(dorange_price)  # (2)
dapple, dapple_num = mul_apple_layer.backward(dapple_price)  # (1)

print("price:", int(price))
print("dApple:", dapple)
print("dApple_num:", int(dapple_num))
print("dOrange:", round(dorange))
print("dOrange_num:", int(dorange_num))
print("dTax:", dtax)

price: 715
dApple: 2.2
dApple_num: 110
dOrange: 3
dOrange_num: 165
dTax: 650


In [ ]:
# ReLUレイヤ
class ReLU:
    def __init__(self):
        self.mask = None

    def forward(self, x):
        self.mask = (x <= 0) # 例） [-1.0,  2.0, -3.0,  4.0]
        out = x.copy()       # 例） [True, False, True, False]
        out[self.mask] = 0   # 例） [ 0.0, 2.0,  0.0, 4.0]
        return out

    def backward(self, dout):
        dout[self.mask] = 0
        dx = dout
        return dx

$$
\frac{\partial y}{\partial x} =
\begin{cases}
1 & (x > 0) \\
0 & (x \le 0)
\end{cases}
$$

In [ ]:
# Sigmoidレイヤ
class Sigmoid:
    def __init__(self):
        self.out = None

    def forward(self):
        out = 1 / (1 + np.exp(-x))
        self.out = out
        return out

    def backward(self):
        dx = dout * (1.0 - self.out) * self.out
        return dx

$$
\begin{aligned}
\frac{\partial y}{\partial x} &= \frac{\exp(-x)}{(1 + \exp(-x))^2} \\
&= \frac{1}{1 + \exp(-x)} \cdot \frac{\exp(-x)}{1 + \exp(-x)} \\
&= y(1 - y)
\end{aligned}
$$

In [ ]:
# Affineレイヤ
# ニューラルネットワークの順伝播で使う行列の積を、幾何学の分野ではアフィン変換を呼ぶ
class Affine:
    def __init__(self, W, b):
        self.W = W
        self.b = b
        self.x = None
        self.dW = None
        self.db = None

    def forward(self, x):
        self.x = x
        out = np.dot(x, self.W) + self.b

    def backward(self):
        dx = np.dot(dout, self.W.T)
        self.dW = np.dot(self.x.T, dout)
        self.db = np.sum(dout, axis=0)
        return dx

In [ ]:
# Softmax-with-Lossレイヤ
# 推論と学習の2つのフェーズがある
#  推論では、最大値にのみ興味があるので、正規化しない出力（スコア）を出すことが多い
#  学習では、正規化する必要があるので、Softmaxレイヤが必要となる

# softmax＋交差エントロピー誤差、恒等関数＋2乗和誤差のような組み合わせだと
# 逆伝播が出力と教師データの差で表現されるようになっている
# ↑こうなるように、損失関数は設計されている

from common.loss import cross_entropy_error

class SoftmaxWithLoss:
    def __init__(self):
        self.loss = None # 損失
        self.y = None    # softmaxの出力
        self.t = None    # 教師データ（one-hot-vector）

    def forward(self, x, t):
        self.t = t
        self.y = softmax(x)
        self.loss = cross_entropy_error(self.y, self.t)
        return self.loss

    def backward(self. dout=1):
        batch_size = self.t.shape[0]
        dx = (self.y - self.t) / batch_size
        return dx